In [1]:
from pathlib import Path
import re
import json
import hashlib
import pandas as pd
import numpy as np
import fitz

In [2]:

from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib as plt


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


DATA_DIR = PROJECT_ROOT / 'data'
BRONZE     = DATA_DIR / 'bronze'
VENTAS     = BRONZE / 'ventas_semanal'
TICKETS    = BRONZE / 'pdfs'
SILVER     = DATA_DIR / 'silver'


print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')

BRONZE_SNAPSHOTS = BRONZE / "snapshots"
BRONZE_SNAPSHOTS.mkdir(parents=True, exist_ok=True)

Project root: /Volumes/SSDExterno/Lauris/Master/TFM/TFM-Hosteleria-AI
Data directory: /Volumes/SSDExterno/Lauris/Master/TFM/TFM-Hosteleria-AI/data


In [3]:

PDF_DIR_EJ = DATA_DIR / "DocTpv-00001TM00000002.pdf"

PDF_DIR_EJ

PosixPath('/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Hosteleria-AI/data/DocTpv-00001TM00000002.pdf')

In [4]:
MONEY_RE = r"\d{1,6}(?:\.\d{3})*,\d{2}"
QTY_RE = r"\d{1,4},\d{3}"
TOKEN_IMPORTE_RE = rf"(?:{MONEY_RE}|INVITA\.?|INVITADO|DTO\.?)"

In [5]:
def normalizar_texto(texto):
    if texto is None:
        return None
    
    texto = str(texto)
    texto = texto.replace("\xa0", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

def numero_es_a_float(valor):
    """
    Convierte números españoles:
    '75,80' -> 75.8
    '1.234,56' -> 1234.56
    'INVITA.' -> np.nan
    """
    if valor is None:
        return np.nan
    
    valor = str(valor).strip().upper()
    
    if valor in ["", "NAN", "NONE"]:
        return np.nan
    
    if "INVITA" in valor or "INVITADO" in valor or "DTO" in valor:
        return 0.0
    
    valor = valor.replace(".", "")
    valor = valor.replace(",", ".")
    
    try:
        return float(valor)
    except ValueError:
        return np.nan   

def cantidad_es_a_float(valor):
    """
    Convierte cantidades:
    '1,000' -> 1.0
    '2,000' -> 2.0
    """
    if valor is None:
        return np.nan
    
    valor = str(valor).strip().replace(",", ".")
    
    try:
        return float(valor)
    except ValueError:
        return np.nan

In [6]:
def extraer_lineas_pdf(pdf_path, y_tolerance=3):
    """
    Extrae líneas de un PDF digital usando coordenadas.
    Es más fiable que page.get_text() para tickets con columnas.
    """
    pdf_path = Path(pdf_path)
    doc = fitz.open(pdf_path)
    
    todas_lineas = []
    
    for page_num, page in enumerate(doc, start=1):
        
        words = page.get_text("words")
        
        if not words:
            continue
        
        palabras = []
        
        for w in words:
            x0, y0, x1, y1, text, block_no, line_no, word_no = w
            
            palabras.append({
                "page": page_num,
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "text": text
            })
        
        palabras = sorted(palabras, key=lambda d: (d["y0"], d["x0"]))
        
        grupos = []
        
        for palabra in palabras:
            asignada = False
            
            for grupo in grupos:
                if abs(palabra["y0"] - grupo["y"]) <= y_tolerance:
                    grupo["words"].append(palabra)
                    grupo["y_values"].append(palabra["y0"])
                    grupo["y"] = np.mean(grupo["y_values"])
                    asignada = True
                    break
            
            if not asignada:
                grupos.append({
                    "y": palabra["y0"],
                    "y_values": [palabra["y0"]],
                    "words": [palabra]
                })
        
        grupos = sorted(grupos, key=lambda g: g["y"])
        
        for grupo in grupos:
            words_ordenadas = sorted(grupo["words"], key=lambda d: d["x0"])
            linea = " ".join(w["text"] for w in words_ordenadas)
            linea = normalizar_texto(linea)
            
            if linea:
                todas_lineas.append(linea)
    
    doc.close()
    
    return todas_lineas

def imprimir_lineas(lineas, max_lineas=None):
    if max_lineas is None:
        max_lineas = len(lineas)
    
    for i, linea in enumerate(lineas[:max_lineas]):
        print(f"{i:03d}: {linea}")

In [7]:
class TicketPDFAgent:
    """
    Agente determinista para extraer tickets de restaurante desde PDFs digitales.
    
    Devuelve:
    - cabecera del ticket
    - líneas de productos
    - totales
    - pagos
    - validaciones
    """
    
    def __init__(self, y_tolerance=3, verbose=False):
        self.y_tolerance = y_tolerance
        self.verbose = verbose
    
    
    def parse_pdf(self, pdf_path):
        pdf_path = Path(pdf_path)
        
        lineas = extraer_lineas_pdf(
            pdf_path,
            y_tolerance=self.y_tolerance
        )
        
        texto = "\n".join(lineas)
        
        cabecera = self.extraer_cabecera(lineas, texto, pdf_path)
        totales = self.extraer_totales(lineas)
        pagos = self.extraer_pagos(lineas)
        items = self.extraer_items(lineas)
        
        validacion = self.validar_ticket(items, totales)
        
        resultado = {
            "archivo_pdf": pdf_path.name,
            "ruta_pdf": str(pdf_path),
            #"hash_pdf": crear_hash_archivo(pdf_path),
            "cabecera": cabecera,
            "totales": totales,
            "pagos": pagos,
            "items": items,
            "validacion": validacion,
            "lineas_texto": lineas
        }
        
        return resultado
    
    
    def extraer_cabecera(self, lineas, texto, pdf_path):
        cabecera = {}
        
        cabecera["restaurante"] = self.extraer_restaurante(lineas)
        cabecera["cif"] = self.extraer_cif(texto)
        cabecera["telefono"] = self.extraer_telefono(texto)
        cabecera["mesa"] = self.extraer_mesa(texto)
        cabecera["ticket_id"] = self.extraer_ticket_id(texto)
        cabecera["fecha"] = self.extraer_fecha(texto)
        cabecera["hora"] = self.extraer_hora(texto)
        cabecera["archivo_origen"] = Path(pdf_path).name
        
        if cabecera["ticket_id"] is None:
            cabecera["ticket_id"] = Path(pdf_path).stem
        
        return cabecera
    
    
    def extraer_restaurante(self, lineas):
        candidatos = []
        
        for linea in lineas[:15]:
            l = linea.upper()
            
            if (
                "S.L" in l
                or "SL" in l
                or "RESTAURANTE" in l
                or "MADRID" in l
            ):
                candidatos.append(linea)
        
        if candidatos:
            return candidatos[0]
        
        return None
    
    
    def extraer_cif(self, texto):
        m = re.search(r"\b[ABCDEFGHJNPQRSUVW]\d{7,8}\b", texto.upper())
        
        if m:
            return m.group(0)
        
        return None
    
    
    def extraer_telefono(self, texto):
        m = re.search(r"\b(?:\d{3}\s?\d{3}\s?\d{3})\b", texto)
        
        if m:
            return normalizar_texto(m.group(0))
        
        return None
    
    
    def extraer_mesa(self, texto):
        patrones = [
            r"Mesa[:\s]+(\d+)",
            r"MESA[:\s]+(\d+)"
        ]
        
        for patron in patrones:
            m = re.search(patron, texto, flags=re.IGNORECASE)
            
            if m:
                return m.group(1)
        
        return None
    
    
    def extraer_ticket_id(self, texto):
        patrones = [
            r"Fra\.?\s*Sim\.?[:\s]+([A-Z0-9]+TM[A-Z0-9]+)",
            r"Factura[:\s]+([A-Z0-9]+TM[A-Z0-9]+)",
            r"\b([A-Z0-9]{4,}TM[A-Z0-9]{4,})\b"
        ]
        
        for patron in patrones:
            m = re.search(patron, texto, flags=re.IGNORECASE)
            
            if m:
                return m.group(1)
        
        return None
    
    
    def extraer_fecha(self, texto):
        m = re.search(r"\b(\d{2}/\d{2}/\d{4})\b", texto)
        
        if m:
            return m.group(1)
        
        return None
    
    
    def extraer_hora(self, texto):
        m = re.search(r"\b(\d{2}:\d{2})\b", texto)
        
        if m:
            return m.group(1)
        
        return None
    
    
    def extraer_totales(self, lineas):
        totales = {
            "base": np.nan,
            "porcentaje_iva": np.nan,
            "iva": np.nan,
            "total": np.nan
        }
        
        # Base / IVA / Total IVA
        for i, linea in enumerate(lineas):
            l = linea.upper()
            
            if "BASE" in l and "IVA" in l:
                
                # Normalmente los importes vienen en la siguiente línea
                for j in range(i + 1, min(i + 5, len(lineas))):
                    importes = re.findall(MONEY_RE, lineas[j])
                    
                    if len(importes) >= 3:
                        totales["base"] = numero_es_a_float(importes[0])
                        totales["porcentaje_iva"] = numero_es_a_float(importes[1])
                        totales["iva"] = numero_es_a_float(importes[2])
                        break
        
        # Total final
        total = self.extraer_total_final(lineas)
        totales["total"] = total
        
        return totales
    
    
    def extraer_total_final(self, lineas):
        """
        Busca el TOTAL real del ticket.
        Evita confundirlo con 'Total IVA'.
        """
        
        for i in range(len(lineas) - 1, -1, -1):
            linea = lineas[i]
            l = linea.upper()
            
            if "TOTAL IVA" in l:
                continue
            
            if "BASE" in l and "IVA" in l:
                continue
            
            if re.search(r"\bTOTAL\b", l):
                importes_misma_linea = re.findall(MONEY_RE, linea)
                
                if importes_misma_linea:
                    return numero_es_a_float(importes_misma_linea[-1])
                
                # Si TOTAL está solo, mirar las siguientes líneas
                for j in range(i + 1, min(i + 4, len(lineas))):
                    importes_siguiente = re.findall(MONEY_RE, lineas[j])
                    
                    if importes_siguiente:
                        return numero_es_a_float(importes_siguiente[-1])
        
        return np.nan
    
    
    def extraer_pagos(self, lineas):
        pagos = {
            "efectivo": np.nan,
            "tarjeta": np.nan
        }
        
        for i, linea in enumerate(lineas):
            l = linea.upper()
            
            if "EFECTIVO" in l:
                importes = re.findall(MONEY_RE, linea)
                
                if importes:
                    pagos["efectivo"] = numero_es_a_float(importes[-1])
                elif i + 1 < len(lineas):
                    importes_sig = re.findall(MONEY_RE, lineas[i + 1])
                    
                    if importes_sig:
                        pagos["efectivo"] = numero_es_a_float(importes_sig[-1])
            
            if "TARJETA" in l:
                importes = re.findall(MONEY_RE, linea)
                
                if importes:
                    pagos["tarjeta"] = numero_es_a_float(importes[-1])
                elif i + 1 < len(lineas):
                    importes_sig = re.findall(MONEY_RE, lineas[i + 1])
                    
                    if importes_sig:
                        pagos["tarjeta"] = numero_es_a_float(importes_sig[-1])
        
        return pagos
    
    
    def extraer_items(self, lineas):
        inicio = self.detectar_inicio_items(lineas)
        fin = self.detectar_fin_items(lineas, inicio)
        
        if inicio is None:
            inicio = 0
        
        if fin is None:
            fin = len(lineas)
        
        lineas_items = lineas[inicio:fin]
        
        items = []
        i = 0
        
        while i < len(lineas_items):
            linea = lineas_items[i]
            
            if self.es_linea_inicio_item(linea):
                
                buffer_item = [linea]
                i += 1
                
                # Captura líneas partidas hasta el siguiente producto o fin de zona
                while i < len(lineas_items):
                    siguiente = lineas_items[i]
                    
                    if self.es_linea_inicio_item(siguiente):
                        break
                    
                    if self.es_linea_stop(siguiente):
                        break
                    
                    buffer_item.append(siguiente)
                    i += 1
                
                item = self.parsear_buffer_item(buffer_item)
                
                if item is not None:
                    items.append(item)
                
            else:
                i += 1
        
        # Añadir número de línea
        for idx, item in enumerate(items, start=1):
            item["linea_item"] = idx
        
        return items
    
    
    def detectar_inicio_items(self, lineas):
        for i, linea in enumerate(lineas):
            l = linea.upper()
            
            if (
                "UNID" in l
                and "DESCRIP" in l
                and ("PRECIO" in l or "IMPORTE" in l)
            ):
                return i + 1
        
        # fallback: primera línea que empieza por cantidad
        for i, linea in enumerate(lineas):
            if self.es_linea_inicio_item(linea):
                return i
        
        return None
    
    
    def detectar_fin_items(self, lineas, inicio):
        if inicio is None:
            inicio = 0
        
        for i in range(inicio, len(lineas)):
            if self.es_linea_stop(lineas[i]):
                return i
        
        return None
    
    
    def es_linea_inicio_item(self, linea):
        linea = linea.strip()
        return re.match(rf"^{QTY_RE}\b", linea) is not None
    
    
    def es_linea_stop(self, linea):
        l = linea.upper().strip()
        
        stops = [
            "BASE",
            "% IVA",
            "TOTAL IVA",
            "TOTAL",
            "GRACIAS",
            "EFECTIVO",
            "TARJETA",
            "WWW.",
            "LE ATENDIÓ",
            "LE ATENDIO",
            "CLIENTE",
            "N.I.F",
            "NOMBRE",
            "DIRECCIÓN",
            "DIRECCION",
            "POBLACIÓN",
            "POBLACION"
        ]
        
        for stop in stops:
            if l.startswith(stop):
                return True
        
        return False
    
    
    def parsear_buffer_item(self, buffer_item):
        texto_item = " ".join(buffer_item)
        texto_item = normalizar_texto(texto_item)
        
        m_qty = re.match(rf"^({QTY_RE})\s+(.*)$", texto_item)
        
        if not m_qty:
            return None
        
        cantidad_txt = m_qty.group(1)
        resto = m_qty.group(2).strip()
        
        # Caso ideal:
        # 1,000 SALMOREJO CON 7,20 7,20
        patron_final = re.compile(
            rf"^(.*?)\s+({TOKEN_IMPORTE_RE})\s+({TOKEN_IMPORTE_RE})$",
            flags=re.IGNORECASE
        )
        
        m = patron_final.match(resto)
        
        if m:
            descripcion = normalizar_texto(m.group(1))
            precio_txt = m.group(2)
            importe_txt = m.group(3)
        else:
            descripcion = normalizar_texto(resto)
            precio_txt = None
            importe_txt = None
        
        item = {
            "cantidad": cantidad_es_a_float(cantidad_txt),
            "descripcion_original": descripcion,
            "producto": self.normalizar_producto(descripcion),
            "precio_unitario": numero_es_a_float(precio_txt),
            "importe": numero_es_a_float(importe_txt),
            "precio_original": precio_txt,
            "importe_original": importe_txt,
            "es_invitacion": self.es_invitacion(precio_txt, importe_txt, descripcion),
            "parse_status": "ok" if m else "sin_precio_importe"
        }
        
        return item
    
    
    def normalizar_producto(self, producto):
        if producto is None:
            return None
        
        producto = str(producto).upper()
        producto = producto.replace(".", " ")
        producto = producto.replace(",", " ")
        producto = re.sub(r"\s+", " ", producto)
        producto = producto.strip()
        
        return producto
    
    
    def es_invitacion(self, precio_txt, importe_txt, descripcion):
        texto = " ".join([
            str(precio_txt or ""),
            str(importe_txt or ""),
            str(descripcion or "")
        ]).upper()
        
        return "INVITA" in texto or "INVITADO" in texto
    
    
    def validar_ticket(self, items, totales):
        df = pd.DataFrame(items)
        
        if df.empty:
            suma_items = np.nan
            num_items = 0
        else:
            suma_items = df["importe"].sum(skipna=True)
            num_items = len(df)
        
        total = totales.get("total", np.nan)
        
        if pd.isna(total) or pd.isna(suma_items):
            diferencia = np.nan
            cuadra_total = False
        else:
            diferencia = round(float(total) - float(suma_items), 2)
            cuadra_total = abs(diferencia) <= 0.05
        
        return {
            "num_items": num_items,
            "suma_items": suma_items,
            "total_ticket": total,
            "diferencia_total_vs_items": diferencia,
            "cuadra_total": cuadra_total
        }

In [8]:
def resultado_a_tablas(resultado):
    cabecera = resultado["cabecera"]
    totales = resultado["totales"]
    pagos = resultado["pagos"]
    validacion = resultado["validacion"]
    ticket_id = cabecera.get("ticket_id")
    
    ticket_row = {
        "ticket_id": ticket_id,
        "archivo_pdf": resultado["archivo_pdf"],
        "ruta_pdf": resultado["ruta_pdf"],
        #"hash_pdf": resultado["hash_pdf"],
        
        "restaurante": cabecera.get("restaurante"),
        "cif": cabecera.get("cif"),
        "telefono": cabecera.get("telefono"),
        "mesa": cabecera.get("mesa"),
        "fecha": cabecera.get("fecha"),
        "hora": cabecera.get("hora"),
        
        "base": totales.get("base"),
        "porcentaje_iva": totales.get("porcentaje_iva"),
        "iva": totales.get("iva"),
        "total": totales.get("total"),
        
        "efectivo": pagos.get("efectivo"),
        "tarjeta": pagos.get("tarjeta"),
        
        "num_items": validacion.get("num_items"),
        "suma_items": validacion.get("suma_items"),
        "diferencia_total_vs_items": validacion.get("diferencia_total_vs_items"),
        "cuadra_total": validacion.get("cuadra_total")
    }
    
    items_rows = []
    
    for item in resultado["items"]:
        row = {
            "ticket_id": ticket_id,
            "archivo_pdf": resultado["archivo_pdf"],
            "fecha": cabecera.get("fecha"),
            "hora": cabecera.get("hora"),
            "mesa": cabecera.get("mesa"),
            **item
        }
        
        items_rows.append(row)
    
    return ticket_row, items_rows

In [9]:
pdfs = list(DATA_DIR.rglob("*.pdf"))
tickets_rows = []
items_rows = []
errores = []
agent = TicketPDFAgent()

for pdf_path in pdfs:
    try:
        resultado = agent.parse_pdf(pdf_path)
        ticket_row, items = resultado_a_tablas(resultado)
        tickets_rows.append(ticket_row)
        items_rows.extend(items)
    except Exception as e:
        errores.append({
            "pdf": str(pdf_path),
            "error": str(e)
        })


In [10]:
df_tickets = pd.DataFrame(tickets_rows)
df_tickets

,ticket_id,archivo_pdf,ruta_pdf,restaurante,cif,telefono,mesa,fecha,hora,base,porcentaje_iva,iva,total,efectivo,tarjeta,num_items,suma_items,diferencia_total_vs_items,cuadra_total
0,00001TM00004601,DocTpv-00001TM00004601.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,211,18/04/2026,16:27,91.68,10.0,9.17,100.85,100.85,NaN,11,100.85,-0.0,True
1,00001TM00006765,DocTpv-00001TM00006765.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,006,04/07/2026,00:11,102.95,10.0,10.30,113.25,120.00,NaN,13,113.25,0.0,True
2,00001TM00000894,DocTpv-00001TM00000894.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,016,04/11/2025,23:46,40.91,10.0,4.09,45.00,45.00,NaN,1,45.00,0.0,True
3,00001TM00001395,DocTpv-00001TM00001395.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,013,27/11/2025,23:32,337.27,10.0,33.73,371.00,376.00,NaN,23,371.00,0.0,True
4,00001TM00006404,DocTpv-00001TM00006404.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,203,20/06/2026,22:29,75.64,10.0,7.56,83.20,83.20,NaN,7,83.20,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,00001TM00000130,DocTpv-00001TM00000130.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,006,05/10/2025,16:56,149.82,10.0,14.98,164.80,170.00,NaN,14,164.80,0.0,True
68,00001TM00000468,DocTpv-00001TM00000468.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,203,19/10/2025,15:59,58.55,10.0,5.85,64.40,64.40,NaN,8,64.40,0.0,True
69,00001TM00001993,DocTpv-00001TM00001993.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,008,21/12/2025,17:25,96.05,10.0,9.60,105.65,105.65,NaN,11,105.65,0.0,True
70,00001TM00000906,DocTpv-00001TM00000906.pdf,/Volumes/SSDExterno/Lauris/Master/TFM/TFM-Host...,LA ROCA MADRID S.L.,B87804415,911 250 564,001,05/11/2025,16:32,184.95,10.0,18.50,203.45,210.00,NaN,10,203.45,0.0,True


In [11]:
def preparar_para_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convierte columnas object con tipos mixtos a string puro.
    Solo afecta al snapshot — no modifica el DataFrame original.
    """
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

In [12]:
# Guardar tickets procesados como Parquet en Bronze/Snapshots

ruta = BRONZE_SNAPSHOTS / "facturas_raw.parquet"

preparar_para_parquet(df_tickets).to_parquet(ruta, index=False)

print(f"✓ tickets_raw.parquet — {len(df_tickets):,} filas x {len(df_tickets.columns)} columnas")

✓ tickets_raw.parquet — 72 filas x 19 columnas


/var/folders/8p/w0tfqpks0xn1c0tbzdbrcdg00000gr/T/ipykernel_94613/1189800153.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
